<a href="https://colab.research.google.com/github/FrankAlvaradoR/c_plusplus/blob/main/Practica_4_prograAvanza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%writefile practica4.cpp

#include <cstdio>
#include <cstdlib>
// agrupa una colección de funciones fundamentales que no encajan en categorías más específicas

// Estructura de datos simulando la lectura de un sensor
struct SensorData {
    int id;
    float temperatura;
    char estado;
};

int main() {
    printf("--- DATALOGGER: PERSISTENCIA Y FORMATOS DE ARCHIVOS ---\n\n");

    const int NUM_LECTURAS = 100;
    SensorData sensor;

    // =========================================================================
    // 1.1 y 1.2 Archivos ASCII y Rutinas de Conversión
    // =========================================================================
    // Abrimos archivo en modo texto ("w"). Requiere conversión de binario a texto legible.
    FILE *archivo_ascii = fopen("datos.csv", "w");
    if (archivo_ascii == NULL) {
        printf("Error al crear el archivo ASCII.\n");
        return 1;
    }

    printf("[SISTEMA]: Escribiendo %d lecturas en formato ASCII (datos.csv)...\n", NUM_LECTURAS);
    for (int i = 0; i < NUM_LECTURAS; i++) {
        sensor.id = i + 1;
        sensor.temperatura = 20.0f + (i * 0.5f);
        sensor.estado = 'A'; // 'A' de Activo

        // Rutina de conversión: traduce los datos en memoria a caracteres ASCII
        fprintf(archivo_ascii, "%d,%.2f,%c\n", sensor.id, sensor.temperatura, sensor.estado);
    }
    fclose(archivo_ascii);

    // =========================================================================
    // 1.4, 1.5 y 1.6 Entrada/Salida Binaria y Control de Buferización
    // =========================================================================
    // Abrimos archivo en modo binario ("wb"). Copia cruda de la memoria, sin conversión.
    FILE *archivo_bin = fopen("datos.bin", "wb");
    if (archivo_bin == NULL) {
        printf("Error al crear el archivo Binario.\n");
        return 1;
    }

    // 1.6 Deshabilitar la buferización por completo (Entrada/Salida no buferizada)
    // Esto asegura que los datos vayan directo al disco, ideal para evitar pérdida de datos
    // en sistemas críticos si hay un corte de energía (Problemas de buferización - 1.5).
    setvbuf(archivo_bin, NULL, _IONBF, 0);

    printf("[SISTEMA]: Escribiendo %d lecturas en formato Binario (datos.bin) sin bufer...\n", NUM_LECTURAS);
    for (int i = 0; i < NUM_LECTURAS; i++) {
        sensor.id = i + 1;
        sensor.temperatura = 20.0f + (i * 0.5f);
        sensor.estado = 'A';

        // E/S Binaria: transfiere el bloque de memoria completo directamente
        fwrite(&sensor, sizeof(SensorData), 1, archivo_bin);

        // Nota: Si no hubieramos usado setvbuf, tendríamos que usar fflush(archivo_bin) aquí.
    }
    fclose(archivo_bin);

    // =========================================================================
    // 1.3 El carácter especial EOF (Lectura Binaria)
    // =========================================================================
    printf("\n[SISTEMA]: Leyendo y verificando archivo binario hasta encontrar EOF...\n");
    FILE *leer_bin = fopen("datos.bin", "rb");

    int leidos = 0;
    SensorData lectura_temp;

    // fread devuelve el número de elementos leídos. Cuando llega al final, devuelve 0.
    // feof() comprueba si el indicador de fin de archivo (EOF) se ha establecido.
    while (fread(&lectura_temp, sizeof(SensorData), 1, leer_bin) == 1) {
        leidos++;
        // Imprimimos solo los primeros 3 para no saturar la consola
        if (leidos <= 3) {
            printf(">> Lectura %d extraida: Temp=%.2f, Estado=%c\n",
                   lectura_temp.id, lectura_temp.temperatura, lectura_temp.estado);
        }
    }

    if (feof(leer_bin)) {
        printf(">> ... \n[SISTEMA]: Se alcanzo el EOF (End Of File) exitosamente. Total leidos: %d\n", leidos);
    }

    fclose(leer_bin);
    printf("\n--- FIN DE LA PRACTICA ---\n");

    return 0;
}

Overwriting practica4.cpp


In [6]:
!g++ practica4.cpp -o practica4
!./practica4

--- DATALOGGER: PERSISTENCIA Y FORMATOS DE ARCHIVOS ---

[SISTEMA]: Escribiendo 100 lecturas en formato ASCII (datos.csv)...
[SISTEMA]: Escribiendo 100 lecturas en formato Binario (datos.bin) sin bufer...

[SISTEMA]: Leyendo y verificando archivo binario hasta encontrar EOF...
>> Lectura 1 extraida: Temp=20.00, Estado=A
>> Lectura 2 extraida: Temp=20.50, Estado=A
>> Lectura 3 extraida: Temp=21.00, Estado=A
>> ... 
[SISTEMA]: Se alcanzo el EOF (End Of File) exitosamente. Total leidos: 100

--- FIN DE LA PRACTICA ---


In [3]:
!ls -lh datos.csv datos.bin

-rw-r--r-- 1 root root 1.2K Sep  9 16:37 datos.bin
-rw-r--r-- 1 root root 1.1K Sep  9 16:37 datos.csv
